# 04 — Model Eğitimi

**Deney Protokolü:**
- **Deney 1:** Tam özellik seti — tüm modeller
- **Deney 2:** Vajrobol'un 5 özelliği — tüm modeller
- **Deney 3:** RF Top-20 özellik — tüm modeller
- **Deney 4:** 10-Fold Cross Validation — en iyi 3 model

**Not:** SVM büyük veriyle yavaştır (~10-20 dk). Zaman sınırı varsa SVM'yi atlayabilirsin.

## 0. Colab Kurulumu

In [ ]:
import os, sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    REPO_DIR = '/content/phishing-url-detection'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/erenterakye/phishing-url-detection.git {REPO_DIR}
    else:
        !git -C {REPO_DIR} pull

    %cd {REPO_DIR}
    !pip install -q -r requirements.txt

    DATA_PATH = '/content/drive/MyDrive/PhiUSIIL_Phishing_URL_Dataset.csv'
else:
    REPO_DIR = '..'
    DATA_PATH = '../data/PhiUSIIL_Phishing_URL_Dataset.csv'

sys.path.insert(0, REPO_DIR)
print(f'IN_COLAB={IN_COLAB}')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.preprocessing import full_preprocessing_pipeline
from src.feature_selection import rf_feature_importance, get_top5_vajrobol
from src.models import get_baseline_models, train_model
from src.evaluation import evaluate_model, compare_models_table, cross_validate_model

sns.set_theme(style='whitegrid')
%matplotlib inline

## 1. Veri Hazırlığı

In [ ]:
X_train, X_test, y_train, y_test, tld_encoder, scaler = full_preprocessing_pipeline(DATA_PATH)
feature_names = list(X_train.columns)
print(f'Train: {X_train.shape} | Test: {X_test.shape}')

## 2. Özellik Setlerini Hesapla

In [ ]:
# Vajrobol-5
vajrobol_5 = [f for f in get_top5_vajrobol() if f in feature_names]
print(f'Vajrobol-5: {vajrobol_5}')

# RF Top-20 (özellik seçimi)
print('\nRF Top-20 hesaplanıyor (birkaç dakika)...')
rf_top20, _ = rf_feature_importance(X_train, y_train, top_n=20)
print(f'RF Top-20: {rf_top20}')

---
## DENEY 1: Tam Özellik Seti

In [ ]:
print('=== DENEY 1: Tam Özellik Seti ===\n')
results_exp1 = {}
trained_exp1 = {}

for name, model in get_baseline_models().items():
    print(f'[{name}] eğitiliyor...')
    model, elapsed = train_model(model, X_train, y_train)
    metrics = evaluate_model(model, X_test, y_test)
    metrics['Train Time (s)'] = round(elapsed, 2)
    results_exp1[name] = metrics
    trained_exp1[name] = model
    print(f'  Acc={metrics["Accuracy"]:.4f} | F1={metrics["F1-Score"]:.4f} | '
          f'Recall={metrics["Recall"]:.4f} | AUC={metrics["ROC-AUC"]:.4f}\n')

df_exp1 = compare_models_table(results_exp1)
print('\n=== DENEY 1 SONUÇLARI ===')
print(df_exp1[['Accuracy','Precision','Recall','F1-Score','ROC-AUC']].to_string())

---
## DENEY 2: Vajrobol'un 5 Özelliği

In [ ]:
print('=== DENEY 2: Vajrobol 5 Özellik ===\n')
results_exp2 = {}
trained_exp2 = {}

X_train_v5 = X_train[vajrobol_5]
X_test_v5  = X_test[vajrobol_5]

for name, model in get_baseline_models().items():
    print(f'[{name}] eğitiliyor (5 özellik)...')
    model, elapsed = train_model(model, X_train_v5, y_train)
    metrics = evaluate_model(model, X_test_v5, y_test)
    metrics['Train Time (s)'] = round(elapsed, 2)
    results_exp2[name] = metrics
    trained_exp2[name] = model
    print(f'  Acc={metrics["Accuracy"]:.4f} | F1={metrics["F1-Score"]:.4f}\n')

df_exp2 = compare_models_table(results_exp2)
print('\n=== DENEY 2 SONUÇLARI ===')
print(df_exp2[['Accuracy','Precision','Recall','F1-Score','ROC-AUC']].to_string())

lr_acc = results_exp2['Logistic Regression']['Accuracy']
print(f'\nHedef: Vajrobol LR Acc ~%99.97 → Elde edilen: {lr_acc:.4f} ({lr_acc*100:.2f}%)')

---
## DENEY 3: RF Top-20 Özellik

In [ ]:
print('=== DENEY 3: RF Top-20 Özellik ===\n')
results_exp3 = {}
trained_exp3 = {}

X_train_rf = X_train[rf_top20]
X_test_rf  = X_test[rf_top20]

for name, model in get_baseline_models().items():
    print(f'[{name}] eğitiliyor (RF Top-20)...')
    model, elapsed = train_model(model, X_train_rf, y_train)
    metrics = evaluate_model(model, X_test_rf, y_test)
    metrics['Train Time (s)'] = round(elapsed, 2)
    results_exp3[name] = metrics
    trained_exp3[name] = model
    print(f'  Acc={metrics["Accuracy"]:.4f} | F1={metrics["F1-Score"]:.4f}\n')

df_exp3 = compare_models_table(results_exp3)
print('\n=== DENEY 3 SONUÇLARI ===')
print(df_exp3[['Accuracy','Precision','Recall','F1-Score','ROC-AUC']].to_string())

---
## DENEY 4: 10-Fold Cross Validation

In [ ]:
print('=== DENEY 4: 10-Fold Cross Validation ===\n')

X_full = pd.concat([X_train, X_test])
y_full = pd.concat([y_train, y_test])

# Deney 1'de en iyi 3 model
best3 = df_exp1.head(3).index.tolist()
print(f'CV için seçilen modeller: {best3}')

cv_results = {}
for name in best3:
    print(f'\n[{name}] 10-Fold CV...')
    model = get_baseline_models()[name]
    cv_results[name] = cross_validate_model(model, X_full, y_full, cv=10)

# Özet tablo
cv_rows = []
for mname, mets in cv_results.items():
    row = {'Model': mname}
    for metric, vals in mets.items():
        row[f'{metric}_mean'] = round(vals['mean'], 4)
        row[f'{metric}_std']  = round(vals['std'],  4)
    cv_rows.append(row)

df_cv = pd.DataFrame(cv_rows).set_index('Model')
print('\n=== DENEY 4 CV SONUÇLARI ===')
print(df_cv.to_string())